In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, text
import gspread
import gspread_dataframe as gd
import os
from dotenv import load_dotenv

# GOOGLE SHEET
# Đường dẫn tới file JSON (đảm bảo tệp tồn tại)
gs = gspread.service_account(r'd:\OneDrive\KDA_Trinh Võ\KDA data\PYTHON_OPERATION\ma_shondo\mashondo.json')

# Mở Google Sheets bằng Google Sheets ID
sht = gs.open_by_key('1AguJlUqyARJ7Y8YYNqg0Y5YaMdtKgJHDWqE5o12quOg')
SHEET1 = 'Xuất bán - STMDT'

# Thông tin kết nối MySQL
# Kết nối MySQL
load_dotenv()

# 🔗 Kết nối MySQL – tạo duy nhất 1 engine dùng xuyên suốt
# Lấy thông tin từ biến môi trường
host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
database = os.getenv("DB_NAME")
port = os.getenv("DB_PORT", 3306)

# Tạo engine MySQL
connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

query_products_template = """
    SELECT 
        ps.product_id AS parent_product_id,
        ps.code AS default_code,               -- Mã sản phẩm cha
        ps.category_id,
        -- Mã sản phẩm con (nếu có), nếu không thì dùng mã cha
        COALESCE(ps2.code, ps.code) AS fdcode,

        COALESCE(ps2.price, ps.price) AS price,

        -- Size nếu là giày dép
        CASE
            WHEN UPPER(COALESCE(c2.name, c1.name)) IN ('SANDALS', 'KID SANDALS', 'KID SNEAKERS', 'SLIDES', 'SNEAKERS') THEN
                CASE 
                    WHEN RIGHT(COALESCE(ps2.code, ps.code), 1) = 'W' THEN CONCAT(LEFT(COALESCE(ps2.code, ps.code), 2), 'W')
                    ELSE LEFT(COALESCE(ps2.code, ps.code), 2)
                END
            ELSE '#'
        END AS size,

        -- Danh mục con
        COALESCE(c1.name, c2.name) AS subcategory,

        -- Danh mục cha
        COALESCE(c2.name, c1.name) AS category,

        -- Ngày launch từ sản phẩm con nếu có, không thì lấy của sản phẩm cha
        COALESCE(ps2.launch_date, ps.launch_date) AS launch_date,

        -- Phân loại sản phẩm
        CASE
            WHEN COALESCE(ps2.launch_date, ps.launch_date) IS NULL 
                AND UPPER(COALESCE(c2.name, c1.name)) IN ('SANDALS', 'KID SANDALS', 'KID SNEAKERS', 'SLIDES', 'SNEAKERS') 
                THEN 'SP CHỜ BÁN'
            WHEN DATEDIFF(CURRENT_DATE(), COALESCE(ps2.launch_date, ps.launch_date)) <= 90 
                THEN 'SP MỚI'
            WHEN UPPER(COALESCE(c2.name, c1.name)) IN ('BAGS', 'ACCESSORIES', 'BRACELETS', 'HATS', 'T-SHIRTS') 
                THEN 'PHỤ KIỆN'
            ELSE 'SP CŨ'
        END AS type_products,
        ps.image
    FROM products ps
    LEFT JOIN products ps2 
        ON ps2.parent_id = ps.external_product_id   -- Ghép sản phẩm con
    LEFT JOIN categories c1 
        ON ps.category_id = c1.external_category_id
    LEFT JOIN categories c2 
        ON c1.parent_id = c2.category_id
    WHERE ps.parent_id IN (-2, -1)                  -- Chỉ lấy sản phẩm cha
    AND ps.code IS NOT NULL;
"""
# Lấy dữ liệu bán hàng từ database
with engine.connect() as conn:
    df_template_fix = pd.read_sql_query(text(query_products_template), conn)

In [2]:
# SALE ECOM 3 THÁNG GẦN NHẤT
# Lấy thông tin từ biến môi trường
host_ecom = os.getenv("DB_HOST_ECOM")
user_ecom = os.getenv("DB_USER_ECOM")
password_ecom = os.getenv("DB_PASSWORD_ECOM")
database_ecom = os.getenv("DB_NAME_ECOM")
port_ecom = os.getenv("DB_PORT_ECOM", 3306)

# Kết nối MySQL
connection_string_ecom = f"mysql+pymysql://{user_ecom}:{password_ecom}@{host_ecom}:{port_ecom}/{database_ecom}"

# Thêm pool_pre_ping=True và connect_args để tăng thời gian chờ
engine_ecom = create_engine(
    connection_string_ecom,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 30}  # tăng timeout từ mặc định (~10s) lên 20s
)


query_sales_90_days_ecom = """
SELECT
    DATE(eo.order_date) AS date_ord,
    SUBSTRING_INDEX(eo.order_id, '_', -1) AS order_id_clean,
    UPPER(os.name) AS source,
    eoi.product_sku,
    eoi.quantity AS qty,
    eoi.price * eoi.quantity AS rvn,
    (eoi.price * eoi.quantity) + eoi.seller_discount + eoi.voucher_seller_discount AS rvn_tag,
    CASE
        WHEN eo.status = 'cancelled' THEN 'Cancelled'
        WHEN eo.status = 'pending'   THEN 'Confirming'
        WHEN eo.status = 'returned'  THEN 'Return and Refund'
        WHEN eo.status = 'completed' THEN 'Success'
        ELSE eo.status
    END AS status
FROM ecommerce_orders eo
JOIN ecommerce_order_items eoi
    ON eoi.external_order_id = eo.external_order_id
JOIN order_source os
    ON eo.order_source_id = os.id
WHERE
    (
        DATE(eo.order_date) BETWEEN '2025-09-28' AND '2026-02-06'
    )
    AND eo.status <> 'cancelled';
"""

# Lấy dữ liệu bán hàng từ database
with engine_ecom.connect() as conn:
    combined_df_ecom = pd.read_sql_query(text(query_sales_90_days_ecom), conn)

In [3]:
worksheet_sale = sht.worksheet(SHEET1)
worksheet_sale.clear()
gd.set_with_dataframe(worksheet_sale, combined_df_ecom)